In [ ]:
import duckdb
import matplotlib.pyplot as plt
import polars as pl
import seaborn as sns

# Set up connection and test query
conn = duckdb.connect("../../data/ff_platform.duckdb",  read_only=True)

# Exploratory Section
First things first, let's determine how we might filter out some of the noise. I'd like to understand what records we should target to remove. Let's understand the low end of targets. 

In [ ]:
conn.sql("""
      SELECT
          targets,
          COUNT(*) as games,
          ROUND(COUNT(*) * 100.0 / SUM(COUNT(*)) OVER(), 1) as pct_of_games
      FROM core.fct_player_game_stats
      WHERE position IN ('WR', 'TE', 'RB')
      GROUP BY targets
      ORDER BY targets
  """).df()

**Ehhhh** &rarr; this is - fine - but not super helpful. This removes even the best players worst weeks - like the best players will have bad weeks. And I want to see that story. Let's spin up another version of this query. this time broken out by season. 

In [ ]:
receiving_games = conn.sql("""
      WITH player_season_totals AS (
          SELECT
              player_id,
              season,
              SUM(targets) as season_targets,
              COUNT(*) as games_played
          FROM core.fct_player_game_stats
          WHERE position IN ('WR', 'TE', 'RB')
            AND season >= 2009
          GROUP BY player_id, season
      ),
      qualified_player_seasons AS (
          SELECT player_id, season
          FROM player_season_totals
          WHERE season_targets >= 40  -- 2 to 3 targets per game
      )
      SELECT
          g.player_id,
          g.season,
          g.week,
          g.position,
          g.team,
          g.targets,
          g.target_share,
          g.receptions,
          g.receiving_yards,
          g.receiving_epa,
          g.wopr,
          g.offense_pct
      FROM core.fct_player_game_stats g
      INNER JOIN qualified_player_seasons q
          ON g.player_id = q.player_id
          AND g.season = q.season
      WHERE g.position IN ('WR', 'TE', 'RB')
        AND g.season >= 2009
      ORDER BY g.player_id, g.season, g.week
  """).pl()

print(f"Shape: {receiving_games.shape}")
print(f"Unique player-seasons: {receiving_games.groupby(['player_id', 'season']).ngroups}")

In [ ]:
receiving_games.groupby('season')['player_id'].nunique().plot(kind='bar', figsize=(12,4), title='Qualifying Players per Season')
plt.ylabel('Players with 40+ targets')
plt.tight_layout()

In [ ]:
### add this cumulative count
receiving_games['game_num'] = receiving_games.groupby(['player_id', 'season']).cumcount() + 1

### teset out a random player
receiving_games[receiving_games['player_id'] == receiving_games['player_id'].iloc[0]].head(10)

### 4 week rolling
receiving_games['rolling_4g_target_share'] = (
      receiving_games
      .groupby(['player_id', 'season'])['target_share']
      .transform(lambda x: x.shift(1).rolling(window=4, min_periods=4).mean())
  )

# Check out a Hot Streak


In [ ]:
receiving_games['rolling_4g_target_share'].filter(pl.col('pos')=='WR').hist(bins=50, figsize=(10,4))
plt.xlabel('4-Game Rolling Target Share')
plt.ylabel('Frequency')
plt.title('Distribution of Rolling Target Share')

In [ ]:
receiving_games['rolling_4g_target_share'].describe()